# Centrales de Abasto nacional — DENUE establishment counts

For every state in Mexico, download the INEGI **DENUE** bulk extract and count the
establishments that DENUE itself documents as belonging to a central de abasto
(`tipoCenCom = "CENTRAL DE ABASTO"`), grouped by `nom_CenCom`.

Method recap: DENUE georeferences stalls inside centrales/mercados/plazas either
around the block or at a single "star" point per nave — but it *tags* each row
with the commercial-complex name and stall number (`num_local`), which is what we
aggregate on here.

In [1]:
import sys, re, unicodedata, zipfile, json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import geopandas as gpd
import folium
from urllib.request import Request, urlopen

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from ceda_pipeline.data_sources import load_scian_rows
from ceda_pipeline.scian import load_scian_catalog, map_scian_codes
from ceda_pipeline.normalize import normalize_text

DENUE_DIR = ROOT / "data/denue"
OUT_DIR = ROOT / "data/outputs"
OUT_DIR.mkdir(exist_ok=True)
SCIAN_XLSX = ROOT / "data/scian/scian_2023_categorias_y_productos.xlsx"

# state file names on the INEGI bulk server; Edomex is split into two parts
DENUE_FILES = {**{f"{i:02d}": [f"denue_{i:02d}_csv.zip"] for i in range(1, 33) if i != 15},
               "15": ["denue_15_1_csv.zip", "denue_15_2_csv.zip"]}
BASE_URL = "https://www.inegi.org.mx/contenidos/masiva/denue/" 

## 1. Download all state extracts (cached)

In [2]:
def fetch(fname):
    dest = DENUE_DIR / fname
    if dest.exists() and dest.stat().st_size > 100_000:
        return dest
    req = Request(BASE_URL + fname, headers={"User-Agent": "ceda-central-markets/0.1"})
    with urlopen(req, timeout=300) as r, dest.open("wb") as f:
        f.write(r.read())
    return dest

tasks = [f for files in DENUE_FILES.values() for f in files]
with ThreadPoolExecutor(max_workers=6) as ex:
    done = list(ex.map(fetch, tasks))
print(len(done), "files ready")

33 files ready


## 2. Filter `CENTRAL DE ABASTO` rows nationwide

In [3]:
cols = ["id","nom_estab","codigo_act","nombre_act","cve_ent","entidad","cve_mun",
        "municipio","tipoCenCom","nom_CenCom","num_local","latitud","longitud","per_ocu"]

def iter_state_rows(fname):
    with zipfile.ZipFile(DENUE_DIR / fname) as z:
        member = next(n for n in z.namelist() if "conjunto_de_datos" in n and n.lower().endswith(".csv"))
        with z.open(member) as fh:
            df = pd.read_csv(fh, usecols=cols, dtype=str, encoding="latin-1")
    return df

frames = []
for ent, files in DENUE_FILES.items():
    for f in files:
        frames.append(iter_state_rows(f))
denue_mx = pd.concat(frames, ignore_index=True)
print(f"{len(denue_mx):,} establishments nationally")

def norm(s):
    return s.fillna("").map(lambda v: normalize_text(str(v)))

denue_mx["tipo_norm"] = norm(denue_mx["tipoCenCom"])
ca = denue_mx[denue_mx["tipo_norm"].str.contains("CENTRAL DE ABASTO", na=False)].copy()
print(f"{len(ca):,} rows tagged CENTRAL DE ABASTO")
ca["tipo_norm"].value_counts()

6,138,075 establishments nationally


25,516 rows tagged CENTRAL DE ABASTO


tipo_norm
CENTRAL DE ABASTO    25516
Name: count, dtype: int64

## 3. Count establishments per central de abasto

Group key: `entidad` + normalized `nom_CenCom`. Note that large complexes (like CEDA
Iztapalapa) are documented under several sub-complex names (naves), so rows are
also rolled up where the name is clearly a CEDA section.

In [4]:
ca["cencom_norm"] = norm(ca["nom_CenCom"]).str.replace(r"\s+", " ", regex=True).str.strip()
ca.loc[ca["cencom_norm"] == "", "cencom_norm"] = "(SIN NOMBRE DE COMPLEJO)"

per_central = (
    ca.groupby(["entidad", "municipio", "cencom_norm"], as_index=False)
      .agg(establishments=("id", "size"),
           star_points=("latitud", lambda s: (s + "," + ca.loc[s.index, "longitud"]).nunique()),
           stall_ids=("num_local", lambda s: s.notna().sum()))
      .sort_values(["entidad", "establishments"], ascending=[True, False])
)
print(f"{len(per_central)} distinct nom_CenCom centrales documented")
per_central.head(20)

586 distinct nom_CenCom centrales documented


,entidad,municipio,cencom_norm,establishments,star_points,stall_ids
13,Aguascalientes,Jesús María,CENTRO DE ABASTOS VINEDOS SAN MARCOS,37,1,37
6,Aguascalientes,Aguascalientes,CENTRO COMERCIAL AGROPECUARIO,18,18,18
8,Aguascalientes,Aguascalientes,GORDITAS JOSE,7,1,7
9,Aguascalientes,Aguascalientes,GORDITAS L GUADALUPANA,7,1,7
3,Aguascalientes,Aguascalientes,CENTRAL DE ABASTOS,6,6,6
1,Aguascalientes,Aguascalientes,AGROTIANGUIS,4,1,4
17,Aguascalientes,Jesús María,VINEDOS SAN MARCOS,4,4,4
0,Aguascalientes,Aguascalientes,AGROPECUARIO,3,3,3
11,Aguascalientes,Jesús María,CENTRAL DE ABASTOS VINEDOS SAN MARCOS,3,3,3
4,Aguascalientes,Aguascalientes,CENTRAL DE ABASTOS DE AGUASCALIENTES,2,2,2


In [5]:
by_state = (
    ca.groupby("entidad", as_index=False)
      .agg(establishments=("id", "size"), centrales=("cencom_norm", "nunique"))
      .sort_values("establishments", ascending=False)
)
by_state

,entidad,establishments,centrales
5,Ciudad de México,6919,48
19,Puebla,4255,54
15,México,3509,52
18,Oaxaca,3466,16
3,Chiapas,1677,20
9,Guanajuato,583,20
22,San Luis Potosí,581,12
11,Hidalgo,565,18
29,Zacatecas,521,9
14,Morelos,393,11


## 4. SCIAN sector mix inside centrales

In [6]:
scian_rows = load_scian_rows(str(SCIAN_XLSX))
catalog = load_scian_catalog(scian_rows, version="2023", version_date="2023-07-19")
ca["codigo_act"] = ca["codigo_act"].astype(str).str.strip()
mapped, unknown = map_scian_codes(ca["codigo_act"].tolist(), catalog)
by_code = {m["code"]: m for m in mapped}
ca["scian_title"] = ca["codigo_act"].map(lambda c: by_code.get(c, {}).get("title"))

def sector_of(code):
    p = int(str(code)[:2])
    for m in ("31-33", "48-49"):
        lo, hi = [int(x) for x in m.split("-")]
        if lo <= p <= hi:
            return m
    return str(code)[:2]
ca["sector"] = ca["codigo_act"].map(sector_of)
ca["sector_title"] = ca["sector"].map(lambda s: catalog[s].title if s in catalog else s)
print("unmapped:", len(unknown))
ca.groupby(["codigo_act","scian_title"], as_index=False).size() \
  .sort_values("size", ascending=False).head(15)

unmapped: 0


,codigo_act,scian_title,size
126,461130,Comercio al por menor de frutas y verduras fre...,8555
74,431130,Comercio al por mayor de frutas y verduras fre...,2310
142,463211,"Comercio al por menor de ropa, excepto de bebé...",969
127,461140,Comercio al por menor de semillas y granos ali...,919
122,461110,"Comercio al por menor en tiendas de abarrotes,...",827
178,466312,Comercio al por menor de plantas y flores natu...,715
123,461121,Comercio al por menor de carnes rojas,689
131,461190,Comercio al por menor de otros alimentos,555
128,461150,"Comercio al por menor de leche, otros producto...",546
296,722518,Restaurantes que preparan otro tipo de aliment...,470


## 5. Map — one point per central, sized by establishment count

In [7]:
ca["lat"] = pd.to_numeric(ca["latitud"], errors="coerce")
ca["lon"] = pd.to_numeric(ca["longitud"], errors="coerce")
import math
centroids = (ca.dropna(subset=["lat","lon"])
               .groupby(["entidad","cencom_norm"], as_index=False)
               .agg(lat=("lat","median"), lon=("lon","median"), n=("id","size")))
m = folium.Map(location=[23.6, -102.5], zoom_start=5, tiles="OpenStreetMap")
for _, r in centroids.iterrows():
    folium.CircleMarker(
        [r.lat, r.lon], radius=3 + math.sqrt(r.n) * 0.8,
        color="#b10026", fill=True, fill_opacity=0.7, weight=1,
        tooltip=f"{r.cencom_norm} — {r.n} negocios ({r.entidad})",
    ).add_to(m)
m

## 6. Export

In [8]:
per_central.to_csv(OUT_DIR / "centrales_abasto_por_establecimientos.csv", index=False)
by_state.to_csv(OUT_DIR / "centrales_abasto_por_estado.csv", index=False)
m.save(OUT_DIR / "centrales_abasto_nacional.html")
print(f"{len(per_central)} centrales -> {OUT_DIR}")

586 centrales -> /home/chonkydev/Documents/Github Repos/CEDA-in-Central-Markets/data/outputs


## Notes

- Counts come from DENUE's own `tipoCenCom`/`nom_CenCom` tags, not spatial
  intersection — a stall is counted if INEGI recorded it as part of the complex.
- `nom_CenCom` spellings vary (`CENTRAL DE ABASTO` vs `CENTRAL DE ABASTOS`,
  sub-complex naves like `HORTALIZAS Y SUBASTA`); the table keeps them distinct —
  review `cencom_norm` variants when a market must be treated as one unit.
- Vintage: current DENUE (May 2026), not year-sliced.